In [1]:
# Milestone 4 · Step 1 — project code + keys from Kaggle Secrets (never printed) + Langfuse check
!git clone -q https://github.com/jamalla/arabic-book-recommender-agent.git repo
!pip install -q -r repo/requirements.txt
import os, sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = secrets.get_secret("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_PUBLIC_KEY"] = secrets.get_secret("Public Key")  # the label used in Kaggle
os.environ["LANGFUSE_BASE_URL"] = "https://cloud.langfuse.com"
sys.path.insert(0, "repo")

from langfuse import get_client
print("Langfuse connected:", get_client().auth_check())

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
Langfuse connected: True


In [2]:
# Step 2 — the fixed test set. First make sure the TRAPS are fair: those books really are NOT in the catalog
import pandas as pd
from bookrec.search import BookSearch, normalize_arabic
from evals.dataset import TEST_QUESTIONS

engine = BookSearch()
names = (engine.books["title"] + " " + engine.books["author"]).map(normalize_arabic)
for word in ["هاري بوتر", "نجيب محفوظ", "بايثون", "برمجة"]:
    print(f"{word!r} appears in a catalog title/author: {names.str.contains(normalize_arabic(word)).any()}")

rows = []
for qid, question, kind, _ in TEST_QUESTIONS:
    best = engine.search(question, k=1)[0]
    rows.append({"id": qid, "type": kind, "question": question, "best search match": best["title"], "score": best["score"]})
pd.DataFrame(rows)

'هاري بوتر' appears in a catalog title/author: False
'نجيب محفوظ' appears in a catalog title/author: False
'بايثون' appears in a catalog title/author: False
'برمجة' appears in a catalog title/author: False


,id,type,question,best search match,score
0,q01,normal,أريد رواية تاريخية,تاريخ العرب القديم,0.150
1,q02,normal,أبحث عن كتاب يساعدني في تربية أطفالي,الامراض العصبية والنفسية عند الاطفال والاولاد,0.161
2,q03,normal,أريد كتاباً مبسطاً عن الفلسفة,فلسفة القانون ورهانات العدالة من رهانات التأسي...,0.163
3,q04,normal,كتاب عن الاقتصاد أو إدارة الأعمال,الأولوية التي يحتلها رأس المال البشري في ظل اق...,0.267
4,q05,normal,أريد أن أفهم السياسة في العالم العربي,الفقه السياسي في الإسلام,0.166
5,q06,normal,كتاب علمي عن الطبيعة أو الكون,موجز تاريخ الكون : من الانفجار الأعظم إلى الاس...,0.180
6,q07,normal,أحب قراءة السير الذاتية والمذكرات,ليس سهلاً أن تكون ملكاً,0.188
7,q08,normal,كتاب عن الفن أو الخط العربي,مرجع الطلاب في الخط العربي,0.351
8,q09,normal,أريد كتاباً عن السيرة النبوية,الأنوار في شمائل النبي المختار (ص),0.234
9,t01,trap,هل عندكم رواية هاري بوتر؟,بوفن بوي وغابة النينجا,0.160


In [3]:
# Step 3 — the BASELINE: prompt v1 on all 12 questions, scored by CODE (invented_books) and an LLM JUDGE (fit)
from evals.run_experiment import run

result = run("v1")
print(result.format())

rows = []
for r in result.item_results:
    scores = {e.name: e.value for e in r.evaluations}
    rows.append({"id": r.item.id.split("-")[-1], "type": r.item.metadata["type"], "question": r.item.input["question"],
                 "invented_books": scores.get("invented_books"), "fit": scores.get("fit")})
baseline = pd.DataFrame(rows).sort_values("id")
baseline

Individual Results: Hidden (12 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: prompt-v1
📋 Run name: prompt-v1 - 2026-10-23T05:26:25.907530Z - Agent gpt-6-luna with system prompt v1 on the 12 fixed questions
12 items
Evaluations:
  • invented_books
  • fit

Average Scores:
  • invented_books: 0.917
  • fit: 0.825

Run Evaluations:
  • avg_fit: 0.825
  • grounded_rate: 0.500
    💭 share of answers with 0 invented books
  • invented_total: 11.000

🔗 Experiment:
   https://cloud.langfuse.com/project/cmvkijorh0aqhad0edvnp5xg2/experiments/results?baseline=6924a2bcb300c2fc


,id,type,question,invented_books,fit
11,q01,normal,أريد رواية تاريخية,0,0.9
10,q02,normal,أبحث عن كتاب يساعدني في تربية أطفالي,0,1.0
9,q03,normal,أريد كتاباً مبسطاً عن الفلسفة,1,0.0
8,q04,normal,كتاب عن الاقتصاد أو إدارة الأعمال,1,1.0
7,q05,normal,أريد أن أفهم السياسة في العالم العربي,0,1.0
6,q06,normal,كتاب علمي عن الطبيعة أو الكون,1,1.0
5,q07,normal,أحب قراءة السير الذاتية والمذكرات,1,1.0
4,q08,normal,كتاب عن الفن أو الخط العربي,2,1.0
3,q09,normal,أريد كتاباً عن السيرة النبوية,0,1.0
2,t01,trap,هل عندكم رواية هاري بوتر؟,0,1.0


In [4]:
# Step 4 — evaluate the EVALUATOR: code says "invented", judge says "fit = 1.0". Who is right? Read the evidence.
from evals.evaluators import recommended_titles, in_catalog

for r in sorted(result.item_results, key=lambda r: r.item.id):
    ev = {e.name: e for e in r.evaluations}
    if not ev["invented_books"].value:
        continue
    print(f"\n{r.item.id.split('-')[-1]}  {r.item.input['question']}   | judge fit={ev['fit'].value}: {ev['fit'].comment}")
    for title in recommended_titles(r.output["answer"]):
        if not in_catalog(title):
            nearest = engine.search(title, k=1)[0]
            print(f"   flagged: «{title}»  →  nearest catalog title: «{nearest['title']}»  (similarity {nearest['score']:.2f})")


q03  أريد كتاباً مبسطاً عن الفلسفة   | judge fit=0.0: The only recommended book is an accessible science overview, not a philosophy introduction, despite the answer acknowledging this mismatch.
   flagged: «موجز تاريخ الكون: من الانفجار الأعظم إلى الاستنساخ البشري»  →  nearest catalog title: «موجز تاريخ الكون : من الانفجار الأعظم إلى الاستنساخ البشري»  (similarity 0.36)

q04  كتاب عن الاقتصاد أو إدارة الأعمال   | judge fit=1.0: All four recommended books are in the catalog and clearly address economics or business.
   flagged: «مبادئ الاقتصاد الجزئي»  →  nearest catalog title: «مبائ الاقتصاد الجزئي»  (similarity 0.30)

q06  كتاب علمي عن الطبيعة أو الكون   | judge fit=1.0: Both recommended books are in the catalog and clearly address science, natural phenomena, or the origins of the universe and life.
   flagged: «موجز تاريخ الكون: من الانفجار الأعظم إلى الاستنساخ البشري»  →  nearest catalog title: «موجز تاريخ الكون : من الانفجار الأعظم إلى الاستنساخ البشري»  (similarity 0.36)

q07  أح

In [5]:
# Step 5 — WHY were real books flagged? Show every character that is NOT an Arabic letter (spaces, punctuation, invisible marks)
def non_letters(s):
    return " ".join(f"U+{ord(c):04X}" for c in s if not ("\u0621" <= c <= "\u064A"))

r04 = next(r for r in result.item_results if r.item.id.endswith("q04"))
flagged = next((t for t in recommended_titles(r04.output["answer"]) if not in_catalog(t)), None)
if flagged:
    real = engine.search(flagged, k=1)[0]["title"]
    print("answer :", repr(flagged), "|", non_letters(flagged))
    print("catalog:", repr(real), "|", non_letters(real))
print("\nSame letters, different spaces/punctuation/invisible marks -> our title matcher is too strict: an EVALUATOR bug.")

answer : 'مبادئ الاقتصاد الجزئي' | U+0020 U+0020
catalog: 'مبائ الاقتصاد الجزئي' | U+0020 U+0020

Same letters, different spaces/punctuation/invisible marks -> our title matcher is too strict: an EVALUATOR bug.


In [6]:
# Step 6 — letter-by-letter diff of the two titles
import difflib
for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, flagged, real).get_opcodes():
    if op != "equal":
        print(f"{op:8s} answer={flagged[i1:i2]!r} ({' '.join(f'U+{ord(c):04X}' for c in flagged[i1:i2])})"
              f"   catalog={real[j1:j2]!r} ({' '.join(f'U+{ord(c):04X}' for c in real[j1:j2])})")

delete   answer='د' (U+062F)   catalog='' ()


In [7]:
# Step 7 — pull the evaluator fix (commit bd4b982) and TEST it on the exact cases we found
!cd repo && git pull -q
import importlib, evals.evaluators, evals.run_experiment
importlib.reload(evals.evaluators); importlib.reload(evals.run_experiment)
from evals.evaluators import in_catalog

cases = {  # title as the agent wrote it -> is it a real catalog book?
    "موجز تاريخ الكون: من الانفجار الأعظم إلى الاستنساخ البشري": True,   # space before ":" in the catalog
    "قرارات مصيرية: حياتي في دهاليز السياسة": True,
    "بدائع الخط العربي - الكتاب الثاني": True,                           # dash spacing
    "موسوعة الخط العربي: الخطوط العربية الأخرى": True,                    # ":" vs "-"
    "مبادئ الاقتصاد الجزئي": True,                                       # typo «مبائ» in the catalog
    "الثلاثية": False, "زقاق المدق": False, "اللص والكلاب": False,       # Naguib Mahfouz, NOT in the catalog
    "ميرامار": False, "أولاد حارتنا": False,
}
passed = 0
for title, expected in cases.items():
    got = in_catalog(title)
    passed += got == expected
    print("✅" if got == expected else "❌", f"in_catalog({title!r}) = {got}   expected {expected}")
print(f"\n{passed}/{len(cases)} evaluator tests pass")

✅ in_catalog('موجز تاريخ الكون: من الانفجار الأعظم إلى الاستنساخ البشري') = True   expected True
✅ in_catalog('قرارات مصيرية: حياتي في دهاليز السياسة') = True   expected True
✅ in_catalog('بدائع الخط العربي - الكتاب الثاني') = True   expected True
✅ in_catalog('موسوعة الخط العربي: الخطوط العربية الأخرى') = True   expected True
✅ in_catalog('مبادئ الاقتصاد الجزئي') = True   expected True
✅ in_catalog('الثلاثية') = False   expected False
✅ in_catalog('زقاق المدق') = False   expected False
✅ in_catalog('اللص والكلاب') = False   expected False
✅ in_catalog('ميرامار') = False   expected False
✅ in_catalog('أولاد حارتنا') = False   expected False

10/10 evaluator tests pass


In [8]:
# Step 8 — the OFFICIAL baseline: prompt v1 again, now scored by the fixed evaluator
result_v1 = evals.run_experiment.run("v1")
print(result_v1.format())

rows = []
for r in result_v1.item_results:
    scores = {e.name: e.value for e in r.evaluations}
    rows.append({"id": r.item.id.split("-")[-1], "type": r.item.metadata["type"], "question": r.item.input["question"],
                 "invented_books": scores.get("invented_books"), "fit": scores.get("fit")})
baseline_v1 = pd.DataFrame(rows).sort_values("id")
print("avg fit:", round(baseline_v1.fit.mean(), 3), "| grounded answers:", (baseline_v1.invented_books == 0).sum(), "/ 12",
      "| invented books:", int(baseline_v1.invented_books.sum()))
baseline_v1

Individual Results: Hidden (12 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: prompt-v1
📋 Run name: prompt-v1 - 2026-10-23T05:32:38.195281Z - Agent gpt-6-luna with system prompt v1 on the 12 fixed questions
12 items
Evaluations:
  • invented_books
  • fit

Average Scores:
  • invented_books: 0.667
  • fit: 0.750

Run Evaluations:
  • avg_fit: 0.750
  • grounded_rate: 0.917
    💭 share of answers with 0 invented books
  • invented_total: 8.000

🔗 Experiment:
   https://cloud.langfuse.com/project/cmvkijorh0aqhad0edvnp5xg2/experiments/results?baseline=b231a53951576e4d
avg fit: 0.75 | grounded answers: 11 / 12 | invented books: 8


,id,type,question,invented_books,fit
11,q01,normal,أريد رواية تاريخية,0,0.50
10,q02,normal,أبحث عن كتاب يساعدني في تربية أطفالي,0,1.00
9,q03,normal,أريد كتاباً مبسطاً عن الفلسفة,0,0.00
8,q04,normal,كتاب عن الاقتصاد أو إدارة الأعمال,0,1.00
7,q05,normal,أريد أن أفهم السياسة في العالم العربي,0,0.75
6,q06,normal,كتاب علمي عن الطبيعة أو الكون,0,1.00
5,q07,normal,أحب قراءة السير الذاتية والمذكرات,0,1.00
4,q08,normal,كتاب عن الفن أو الخط العربي,0,1.00
3,q09,normal,أريد كتاباً عن السيرة النبوية,0,0.75
2,t01,trap,هل عندكم رواية هاري بوتر؟,0,1.00
